# Fig. 4c — Biological processes per VGN metacluster (balanced consensus NMF)

Bubble plot of top GO:BP enrichment terms per VGN metacluster (3-MC scheme: **Lypd1+**, **Vcan+**, **Lmo3+**). Programs are derived from a **balanced consensus NMF (cNMF)** pipeline that:
1. Caps each VGN subtype at `CAP_PER_SUBTYPE` cells to remove abundance bias.
2. Runs cNMF `N_BOOTSTRAP` times with different cell resamplings.
3. Keeps only programs that are stable across bootstraps, broadly shared across subtypes within their dominant metacluster, and pass a 4-axis QC blacklist (stress / mito-ribo / glia / metabolic).
4. Averages the spectra of retained programs within each metacluster and runs GO:BP enrichment on the top 100 genes.

## Stages (skip earlier stages by checking output existence)

| Stage | Output | Cost |
| --- | --- | --- |
| 1. Run cNMF × 10 bootstraps | `cnmf_balanced_3MC/balanced_boot*/` | ~1–2 h |
| 2. Subtype-sharing + 4-axis QC + covariate ρ | `good_programs` DataFrame | seconds |
| 3. Identity signatures per MC | `mc_gene_sigs` dict | seconds |
| 4. GO:BP enrichment + bubble plot | `Fig4c_GO_BP_bubble.pdf` | ~1 min |

## Inputs
- `Vestibular_VGN.h5ad` — VGN AnnData with `meta_celltype` ∈ {Lypd1+, Vcan+, Lmo3+} and `all_cluster_gene_name` in `.obs`.

## Outputs
- `figures/Fig4c_GO_BP_bubble.pdf` — main panel
- `figures/Fig4c_good_programs.csv` — retained programs + per-axis QC
- `figures/Fig4c_mc_gene_sigs.csv` — top-100 signature genes per MC
- `figures/Fig4c_GO_BP_terms.csv` — full GO:BP table for reviewer reference

## Reviewer caveats
- **cNMF is stochastic** — bootstrap seeds are fixed (`seed = 42 + boot_i`) for reproducibility, but the K=11 choice was made from a K-selection plot on a single balanced sampling (cell with `if False:` guard).
- **4-axis QC blacklist** is curated for VGN biology: stress is the 4-gene lab IEG panel `[Fos, Fosb, Jun, Egr1]` only (per lab convention); glia covers Schwann / SGC; metabolic only flags glycolysis/TCA (OXPHOS is the vulnerability axis, NOT contamination — see Fig 2h–l).
- **Covariate ρ filter is informational, not a hard gate** (Path D in the original notebook). Top-gene evidence showed cells failing covariate ρ but passing 4-axis QC are real subtype identity programs; the ρ vs total_counts mostly reflects cell-pool differences across MCs.
- **GO:BP only** in this panel. A complementary GO + REAC + KEGG version with category quotas (axon guidance / cytoskeleton / adhesion / ion channel / metabolic / synaptic) is in the upstream notebook (`meta_celltype_simple_3MC_0520.ipynb` cell 20) and saves `GO_REAC_KEGG_balanced_cNMF_bubble_3MC_0520.pdf`.

In [ ]:
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import entropy, kruskal, spearmanr

warnings.filterwarnings('ignore')
sc.settings.verbosity = 1
sc.set_figure_params(figsize=(5, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.family'] = 'Arial'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

WORK_DIR_BAL = DATA_DIR / 'cnmf_balanced_3MC'
WORK_DIR_BAL.mkdir(exist_ok=True)

## Shared parameters and metadata

In [ ]:
META_ORDER = ['Lypd1+', 'Vcan+', 'Lmo3+']
META_COLORS = {'Lypd1+': '#E64B35', 'Vcan+': '#4DBBD5', 'Lmo3+': '#00A087'}

# cNMF
SELECTED_K = 11      # chosen via K-selection plot (K-selection cell is `if False:` guarded)
N_ITER = 200          # NMF iterations per K per bootstrap
N_TOP_GENES = 2000    # overdispersed genes selected by cNMF.prepare
DENSITY_THRESH = 0.1  # consensus density threshold
N_BOOTSTRAP = 10
CAP_PER_SUBTYPE = 100
RUN_PREFIX = 'balanced'
dt_str = str(DENSITY_THRESH).replace('.', '_')

# Gene blacklist applied at cNMF prep (Schwann myelin) — these dominate variance
# without being biologically VGN identity, so we drop them before factorisation.
MYELIN_GENES = [
    'Mbp', 'Pmp22', 'Plekhb1', 'Mpz', 'Fxyd6', 'Spacdr',
    'Ndrg1', 'Cryab', 'Glul', 'Art3', 'Gpm6b', 'Gpr37l1',
    'Mal', 'Bcas1', 'Cnp',
]

# 4-axis QC blacklists (applied to top-50 genes per program after cNMF)
STRESS_GENES = ['Fos', 'Fosb', 'Jun', 'Egr1']  # lab 4-gene IEG panel (van den Brink 2017 core)
GLIA_GENES = [
    # Schwann / SGC core (myelin & ensheathment)
    'Mpz', 'Mpzl1', 'Mpzl2', 'Pmp22', 'Mbp', 'Plp1', 'Mag', 'Prx', 'Cldn19',
    'Gjb1', 'Gjc3', 'Ncmap', 'Drp2', 'Pllp', 'Egr2', 'Sox10', 'Fabp7', 'S100b',
    'Kcnj10', 'Apod', 'Mlip',
    # Glia/fibroblast supporting (flag if many co-occur)
    'Sparc', 'Serpine2', 'Serpinh1', 'Plpp3', 'Selenop', 'Igfbp6', 'Cd82', 'Aif1l',
]
# OXPHOS (Atp5*/Cox*/Nduf*/Uqcr*) deliberately excluded — those are the vulnerability
# axis in Fig 2h-l, not contamination. Only glycolysis/TCA flagged here.
METABOLIC_GENES = [
    'Ldha', 'Ldhb', 'Pgam1', 'Eno1', 'Eno1b', 'Pkm', 'Tpi1', 'Aldoa', 'Gapdh', 'Pfkp',
    'Mdh1', 'Mdh2',
]

N_TOP_QC = 50         # top-N genes per program inspected for blacklist hits
THRESH_FAIL = 15      # >15% blacklist hit -> FAIL (PI recommendation)
THRESH_WARN = 5       # 5-15% -> WARN

# Subtype-sharing classification thresholds (Step 4)
SHARED_ENTROPY = 0.6   # > -> 'shared'
PARTIAL_ENTROPY = 0.3  # > -> 'partial', else 'subtype-specific'
SHARED_ACTIVE_RATIO = 0.5
PARTIAL_ACTIVE_RATIO = 0.3

# Covariate ρ threshold (Step 5b; informational only — see Path D note)
RHO_THRESHOLD = 0.3

# GO bubble plot
N_TOP_SIG_GENES = 100      # genes per MC fed to gprofiler
N_TERMS_PER_MC = 5         # max terms per MC in bubble plot
MAX_JACCARD = 0.5          # drop term if leading-edge gene-set Jaccard > this with already-picked term

## 1. Load VGN adata and inspect 3-MC mapping

The 3-MC mapping is **already in `adata.obs['meta_celltype']`** — established upstream from `Lypd1`/`Vcan`/`Lmo3` marker expression. Subtypes that were originally `Sall3+` in the 4-MC scheme are folded into one of the three (Calb2 → Lypd1+, Aldh1a3 → Vcan+, Nxph4 → Lmo3+).

In [ ]:
adata_neuron = sc.read_h5ad(DATA_DIR / 'Vestibular_VGN.h5ad')
print(adata_neuron)
print('\nmeta_celltype distribution:')
print(adata_neuron.obs['meta_celltype'].value_counts())
print('\nsubtype × meta_celltype crosstab:')
print(pd.crosstab(adata_neuron.obs['all_cluster_gene_name'], adata_neuron.obs['meta_celltype'])[META_ORDER])

## 2. Stage 1 — Balanced cNMF × 10 bootstraps (~1–2 h)

Each bootstrap:
1. Cap each subtype at `CAP_PER_SUBTYPE` cells (different random subsample per bootstrap, seed = 42 + boot_i).
2. Switch X to raw counts (`umi` layer), apply myelin / mt-* / Rpl* / Rps* gene blacklist.
3. Run cNMF (`prepare → factorize → combine → consensus`) at `SELECTED_K=11`.

Skip the entire stage if all 10 boot directories already exist.

In [ ]:
def _all_bootstraps_done():
    for boot_i in range(N_BOOTSTRAP):
        run_name = f'{RUN_PREFIX}_boot{boot_i}'
        usage_file = WORK_DIR_BAL / run_name / f'{run_name}.usages.k_{SELECTED_K}.dt_{dt_str}.consensus.txt'
        if not usage_file.exists():
            return False
    return True

if _all_bootstraps_done():
    print(f'All {N_BOOTSTRAP} bootstraps found in {WORK_DIR_BAL} — skipping Stage 1.')
else:
    from cnmf import cNMF

    obs = adata_neuron.obs[['all_cluster_gene_name', 'meta_celltype']].copy()
    obs['barcode'] = obs.index

    for boot_i in range(N_BOOTSTRAP):
        seed = 42 + boot_i
        np.random.seed(seed)

        # Resample balanced cells
        balanced_idx = []
        for mc in META_ORDER:
            mc_obs = obs[obs['meta_celltype'] == mc]
            for st in mc_obs['all_cluster_gene_name'].unique():
                st_bc = mc_obs[mc_obs['all_cluster_gene_name'] == st]['barcode'].values
                n_sample = min(len(st_bc), CAP_PER_SUBTYPE)
                balanced_idx.extend(np.random.choice(st_bc, size=n_sample, replace=False))

        # Prepare AnnData with raw counts, gene blacklist applied
        ad_boot = adata_neuron[balanced_idx].copy()
        ad_boot.X = ad_boot.layers['umi'].copy()
        gene_mask = (
            ~ad_boot.var_names.isin(MYELIN_GENES) &
            ~ad_boot.var_names.str.startswith('mt-') &
            ~ad_boot.var_names.str.startswith('Rpl') &
            ~ad_boot.var_names.str.startswith('Rps')
        )
        ad_boot = ad_boot[:, gene_mask].copy()
        sc.pp.filter_genes(ad_boot, min_cells=3)

        run_name = f'{RUN_PREFIX}_boot{boot_i}'
        h5ad_path = str(WORK_DIR_BAL / f'{run_name}.h5ad')
        ad_boot.write(h5ad_path)

        print(f'\n=== Bootstrap {boot_i}/{N_BOOTSTRAP - 1}  '
              f'(seed={seed}, cells={ad_boot.n_obs}, genes={ad_boot.n_vars}) ===')

        cnmf_obj = cNMF(output_dir=str(WORK_DIR_BAL), name=run_name)
        cnmf_obj.prepare(
            counts_fn=h5ad_path,
            components=np.array([SELECTED_K]),
            n_iter=N_ITER, seed=seed, num_highvar_genes=N_TOP_GENES,
        )
        cnmf_obj.factorize(worker_i=0, total_workers=1)
        cnmf_obj.combine()
        cnmf_obj.consensus(k=SELECTED_K, density_threshold=DENSITY_THRESH)

    print(f'\nAll {N_BOOTSTRAP} bootstraps complete.')

## 3. Stage 2 — Subtype-sharing + 4-axis QC + covariate diagnostics

Use bootstrap 0 as the representative run (Step 3 in the upstream notebook confirms modules are stable across bootstraps; the bootstrap loop above is the stability evidence).

Three filters per program:
1. **Subtype sharing** (cell `Step 4`): within the dominant MC, is the program's usage broadly shared across subtypes or driven by one subtype only? `shared` / `partial` pass; `subtype-specific` fails.
2. **4-axis blacklist QC** (cell `Step 5`): % of top-50 genes hitting stress / mito-ribo / glia / metabolic. `WARN` (5–15%) or `PASS` (<5%) keeps the program; `FAIL` (>15%) drops it.
3. **Covariate Spearman ρ** (cell `Step 5b`): |ρ| vs `pct_mt`, `total_counts`, `n_genes_by_counts`, `S_score`, `G2M_score`. **Informational only** in Path D — see reviewer caveats above.

In [ ]:
boot0_dir = WORK_DIR_BAL / f'{RUN_PREFIX}_boot0'
usage_file = boot0_dir / f'{RUN_PREFIX}_boot0.usages.k_{SELECTED_K}.dt_{dt_str}.consensus.txt'
spectra_file = boot0_dir / f'{RUN_PREFIX}_boot0.gene_spectra_score.k_{SELECTED_K}.dt_{dt_str}.txt'

usage_boot0 = pd.read_csv(usage_file, sep='\t', index_col=0)
spectra_boot0 = pd.read_csv(spectra_file, sep='\t', index_col=0)
spectra_boot0.index = spectra_boot0.index.astype(str)

usage_norm = usage_boot0.div(usage_boot0.sum(axis=1), axis=0)
common_cells = usage_norm.index.intersection(adata_neuron.obs_names)
usage_aligned = usage_norm.loc[common_cells]
meta_aligned = adata_neuron.obs.loc[common_cells, ['meta_celltype', 'all_cluster_gene_name']]
print(f'Boot0: {len(usage_aligned)} cells × {usage_aligned.shape[1]} programs')

In [ ]:
# === Step 4: subtype-sharing per program ===
val_rows = []
for prog in usage_aligned.columns:
    mc_means = {
        mc: usage_aligned.loc[meta_aligned['meta_celltype'] == mc, prog].mean()
        for mc in META_ORDER
    }
    dominant_mc = max(mc_means, key=mc_means.get)
    mc_mask = meta_aligned['meta_celltype'] == dominant_mc
    mc_data = pd.DataFrame({
        'usage': usage_aligned.loc[mc_mask, prog].values,
        'subtype': meta_aligned.loc[mc_mask, 'all_cluster_gene_name'].values,
    })
    subtype_means = mc_data.groupby('subtype', observed=True)['usage'].mean()
    subtype_fracs = subtype_means / subtype_means.sum()
    sharing_entropy = entropy(subtype_fracs)
    max_entropy = np.log(len(subtype_means))
    norm_entropy = sharing_entropy / max_entropy if max_entropy > 0 else 0

    groups = [g['usage'].values for _, g in mc_data.groupby('subtype', observed=True) if len(g) >= 3]
    kw_pval = kruskal(*groups)[1] if len(groups) >= 2 else 1.0

    threshold = subtype_means.max() * 0.5
    n_active = (subtype_means > threshold).sum()
    active_ratio = n_active / len(subtype_means)

    if norm_entropy > SHARED_ENTROPY and active_ratio >= SHARED_ACTIVE_RATIO:
        sharing = 'shared'
    elif norm_entropy > PARTIAL_ENTROPY and active_ratio >= PARTIAL_ACTIVE_RATIO:
        sharing = 'partial'
    else:
        sharing = 'subtype-specific'

    val_rows.append({
        'Program': str(prog), 'Dominant_MC': dominant_mc,
        'MC_mean_usage': mc_means[dominant_mc],
        'N_subtypes': len(subtype_means), 'N_active_subtypes': int(n_active),
        'Norm_entropy': norm_entropy, 'KW_pval': kw_pval,
        'Subtype_sharing': sharing,
    })

val_df = pd.DataFrame(val_rows)
print(val_df.to_string(index=False))

In [ ]:
# === Step 5: 4-axis blacklist QC (per program) ===
def axis_status(pct):
    return 'FAIL' if pct > THRESH_FAIL else ('WARN' if pct > THRESH_WARN else 'PASS')

qc_rows = []
for prog in spectra_boot0.index:
    top = spectra_boot0.loc[prog].sort_values(ascending=False).head(N_TOP_QC).index.tolist()
    n_stress = sum(1 for g in top if g in STRESS_GENES)
    n_mito = sum(1 for g in top if g.startswith('mt-'))
    n_ribo = sum(1 for g in top if g.startswith('Rpl') or g.startswith('Rps'))
    n_glia = sum(1 for g in top if g in GLIA_GENES)
    n_metab = sum(1 for g in top if g in METABOLIC_GENES)

    pct = lambda n: n / N_TOP_QC * 100
    statuses = [axis_status(pct(n_stress)), axis_status(pct(n_mito + n_ribo)),
                axis_status(pct(n_glia)), axis_status(pct(n_metab))]
    overall = 'FAIL' if 'FAIL' in statuses else ('WARN' if 'WARN' in statuses else 'PASS')

    qc_rows.append({
        'Program': str(prog),
        'pct_stress': pct(n_stress), 'pct_mitorib': pct(n_mito + n_ribo),
        'pct_glia': pct(n_glia), 'pct_metab': pct(n_metab),
        'Stress_status': statuses[0], 'MitoRib_status': statuses[1],
        'Glia_status': statuses[2], 'Metab_status': statuses[3],
        'QC_status': overall,
    })

qc_df = pd.DataFrame(qc_rows)
print(qc_df[['Program', 'pct_stress', 'pct_mitorib', 'pct_glia', 'pct_metab', 'QC_status']].to_string(index=False))

In [ ]:
# === Step 5b: covariate Spearman ρ (informational) ===
S_GENES_TIROSH = ['Mcm5','Pcna','Tyms','Fen1','Mcm2','Mcm4','Rrm1','Ung','Gins2','Mcm6','Cdca7',
                  'Dtl','Prim1','Uhrf1','Cenpu','Hells','Rfc2','Rpa2','Nasp','Rad51ap1','Gmnn',
                  'Wdr76','Slbp','Ccne2','Ubr7','Pold3','Msh2','Atad2','Rad51','Rrm2','Cdc45',
                  'Cdc6','Exo1','Tipin','Dscc1','Blm','Casp8ap2','Usp1','Clspn','Pola1','Chaf1b',
                  'Brip1','E2f8']
G2M_GENES_TIROSH = ['Hmgb2','Cdk1','Nusap1','Ube2c','Birc5','Tpx2','Top2a','Ndc80','Cks2','Nuf2',
                    'Cks1b','Mki67','Tmpo','Cenpf','Tacc3','Pimreg','Smc4','Ccnb2','Ckap2l','Ckap2',
                    'Aurkb','Bub1','Kif11','Anp32e','Tubb4b','Gtse1','Kif20b','Hjurp','Cdca3','Jpt1',
                    'Cdc20','Ttk','Cdc25c','Kif2c','Rangap1','Ncapd2','Dlgap5','Cdca2','Cdca8','Ect2',
                    'Kif23','Hmmr','Aurka','Psrc1','Anln','Lbr','Ckap5','Cenpe','Ctcf','Nek2','G2e3',
                    'Gas2l3','Cbx5','Cenpa']

if 'S_score' not in adata_neuron.obs.columns or 'G2M_score' not in adata_neuron.obs.columns:
    s_genes = [g for g in S_GENES_TIROSH if g in adata_neuron.var_names]
    g2m_genes = [g for g in G2M_GENES_TIROSH if g in adata_neuron.var_names]
    sc.tl.score_genes_cell_cycle(adata_neuron, s_genes=s_genes, g2m_genes=g2m_genes)

covariates = pd.DataFrame(index=usage_aligned.index)
for col in ['pct_counts_mt', 'total_counts', 'n_genes_by_counts', 'S_score', 'G2M_score']:
    if col in adata_neuron.obs.columns:
        covariates[col.replace('pct_counts_', 'pct_')] = adata_neuron.obs.loc[usage_aligned.index, col].values

cov_rows = []
for prog in usage_aligned.columns:
    rhos = {cov: spearmanr(usage_aligned[prog].values, covariates[cov].values)[0] for cov in covariates.columns}
    max_abs = max(abs(v) for v in rhos.values())
    worst = max(rhos, key=lambda k: abs(rhos[k]))
    cov_rows.append({
        'Program': str(prog),
        **{f'rho_{k}': v for k, v in rhos.items()},
        'max_abs_rho': max_abs, 'worst_cov': worst,
        'Cov_status': 'FAIL' if max_abs > RHO_THRESHOLD else 'PASS',
    })
cov_df = pd.DataFrame(cov_rows)
print(cov_df[['Program', 'max_abs_rho', 'worst_cov', 'Cov_status']].to_string(index=False))

In [ ]:
# Path D: keep programs passing subtype-sharing + 4-axis QC. Cov_status logged but not enforced.
combined = val_df.merge(qc_df, on='Program').merge(cov_df, on='Program')
good_programs = combined[
    combined['Subtype_sharing'].isin(['shared', 'partial']) &
    combined['QC_status'].isin(['PASS', 'WARN'])
].copy()

good_programs.to_csv(FIG_DIR / 'Fig4c_good_programs.csv', index=False)
print(f'{len(good_programs)} / {len(combined)} programs retained')
print(good_programs[['Program', 'Dominant_MC', 'Subtype_sharing', 'QC_status',
                     'max_abs_rho', 'worst_cov', 'Cov_status']].to_string(index=False))

retained_progs = good_programs['Program'].tolist()
mc_to_progs = {mc: good_programs[good_programs['Dominant_MC'] == mc]['Program'].tolist() for mc in META_ORDER}

## 4. Stage 3 — Per-MC identity signatures

For each MC, average the gene-spectra scores of retained programs and take the top `N_TOP_SIG_GENES` (= 100) — these become the input gene list for GO:BP enrichment.

In [ ]:
mc_gene_sigs = {}
for mc in META_ORDER:
    progs = mc_to_progs[mc]
    if not progs:
        print(f'  {mc}: no retained programs')
        continue
    mean_spectra = spectra_boot0.loc[progs].mean(axis=0)
    top_genes = mean_spectra.sort_values(ascending=False).head(N_TOP_SIG_GENES).index.tolist()
    mc_gene_sigs[mc] = top_genes
    print(f'\n{mc} ({len(progs)} programs) — top 20 of {N_TOP_SIG_GENES} signature genes:')
    print(f'  {", ".join(top_genes[:20])}')

pd.DataFrame({mc: pd.Series(g) for mc, g in mc_gene_sigs.items()}).to_csv(
    FIG_DIR / 'Fig4c_mc_gene_sigs.csv', index=False,
)

## 5. Stage 4 — GO:BP enrichment + bubble plot (Fig 4c output)

Per MC:
1. `gprofiler.profile` with the 100-gene signature, custom background = genes detected in ≥10 cells of the full adata.
2. Drop generic root GO terms (e.g. "biological_process") and irrelevant organ terms (kidney / heart / muscle / liver / bone / gonad ...).
3. Tiered `term_size` cut: try 15–500 first; relax to 15–1000 then 15–2000 until at least `N_TERMS_PER_MC` survive.
4. Greedy diversity filter: drop terms whose leading-edge gene set has Jaccard > `MAX_JACCARD` with an already-selected term.
5. Take top `N_TERMS_PER_MC` (= 5) terms per MC.

In [ ]:
from gprofiler import GProfiler

# Background = genes detected in ≥10 cells of the full adata (gp custom domain)
if 'umi' in adata_neuron.layers:
    bg_genes = adata_neuron.var_names[
        np.array((adata_neuron.layers['umi'] > 0).sum(axis=0)).flatten() >= 10
    ].tolist()
else:
    bg_genes = adata_neuron.var_names.tolist()

BLACKLIST_KW = [
    'mesonephr', 'metanephr', 'nephron', 'ureteric', 'kidney', 'renal',
    'cardiac', 'heart', 'muscle', 'liver', 'hepat', 'lung', 'pulmon',
    'bone', 'ossif', 'uterus', 'uterine', 'gonad', 'genital',
    'semi-lunar valve', 'ventricle development',
    'aortic', 'valve', 'sex differentiat', 'sex determin',
    'sexual character', 'osteoclast',
]
GENERIC_TERMS = {
    'biological_process', 'cellular process', 'metabolic process',
    'biological regulation', 'regulation of biological process',
    'regulation of cellular process', 'response to stimulus',
    'multicellular organismal process', 'developmental process',
    'cell communication', 'signaling', 'signal transduction',
    'cellular metabolic process', 'organic substance metabolic process',
    'regulation of metabolic process', 'cellular response to stimulus',
    'positive regulation of biological process',
    'negative regulation of biological process',
    'regulation of biological quality',
}

def passes_filter(name):
    low = name.lower()
    if any(kw in low for kw in BLACKLIST_KW):
        return False
    return low not in GENERIC_TERMS

gp = GProfiler(return_dataframe=True)
all_go_rows = []

for mc in META_ORDER:
    if mc not in mc_gene_sigs:
        continue
    res = gp.profile(
        organism='mmusculus', query=mc_gene_sigs[mc],
        sources=['GO:BP'], user_threshold=0.05,
        background=bg_genes, domain_scope='custom',
        no_evidences=False,  # need 'intersections' for Jaccard diversity filter
    )
    if res.empty:
        print(f'{mc}: no significant GO terms'); continue

    # Robust p-value column rename
    for cand in ['p_value', 'pvalue', 'P-value', 'p-value', 'adjusted_p_value']:
        if cand in res.columns:
            res = res.rename(columns={cand: 'p_value'}); break

    n_raw = len(res)
    res = res[res['term_size'] >= 15]
    res = res[res['name'].apply(passes_filter)]

    # Tiered term_size cutoff (15-500 -> 15-1000 -> 15-2000)
    for max_ts in [500, 1000, 2000]:
        cut = res[res['term_size'] <= max_ts]
        if len(cut) >= N_TERMS_PER_MC:
            res = cut; break
    else:
        res = cut
    print(f'{mc}: {n_raw} raw -> {len(res)} after filters')

    # Greedy Jaccard diversity filter
    res = res.sort_values('p_value').reset_index(drop=True)
    selected_idx, selected_genes = [], []
    for idx, row in res.iterrows():
        current = set(row.get('intersections', []) if isinstance(row.get('intersections', None), list)
                      else [row['native']])
        if any(len(sg & current) / max(len(sg | current), 1) > MAX_JACCARD for sg in selected_genes):
            continue
        selected_idx.append(idx); selected_genes.append(current)
        if len(selected_idx) >= N_TERMS_PER_MC:
            break
    res = res.loc[selected_idx]

    for _, row in res.iterrows():
        all_go_rows.append({
            'MC': mc, 'Term_id': row['native'], 'Term': row['name'],
            'p_value': row['p_value'], 'n_genes': row['intersection_size'],
        })

go_balanced_df = pd.DataFrame(all_go_rows)
go_balanced_df.to_csv(FIG_DIR / 'Fig4c_GO_BP_terms.csv', index=False)
go_balanced_df

In [ ]:
# Tunable bubble plot parameters (copied from the upstream notebook, cell 17)
DOT_SIZE_SCALE = 18
ROW_HEIGHT_INCH = 0.1
DOT_ALPHA = 0.85
FIG_WIDTH_INCH = 7.0
RIGHT_PLOT_FRAC = 0.78
X_PADDING_FRAC = 0.04

from matplotlib.lines import Line2D

n_terms = max(4, len(go_balanced_df) * ROW_HEIGHT_INCH)
fig, ax = plt.subplots(figsize=(FIG_WIDTH_INCH, n_terms))

pdf = pd.DataFrame([
    {
        'MC': row['MC'],
        'Term': row['Term'][:50] + '...' if len(row['Term']) > 50 else row['Term'],
        'nlp': -np.log10(max(row['p_value'], 1e-300)),
        'Genes': row['n_genes'],
    } for _, row in go_balanced_df.iterrows()
])

for mc in META_ORDER:
    sub = pdf[pdf['MC'] == mc]
    if len(sub) == 0:
        continue
    ax.scatter(sub['nlp'], sub['Term'], s=sub['Genes'] * DOT_SIZE_SCALE,
               c=META_COLORS[mc], label=mc, alpha=DOT_ALPHA,
               edgecolors='white', linewidth=0.6, zorder=3)

ax.set_xlabel(r'$-\log_{10}$(p-value)', fontsize=11)
ax.axvline(-np.log10(0.05), color='grey', ls='--', lw=0.8, alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_title('GO:BP — Balanced cNMF', fontsize=12, fontweight='bold')
ax.margins(x=X_PADDING_FRAC, y=0.04)

leg1 = ax.legend(title='Meta-cluster', bbox_to_anchor=(.95, 1),
                 loc='upper left', frameon=False, fontsize=9)
ax.add_artist(leg1)

gc = pdf['Genes'].values
refs = sorted(set([int(np.percentile(gc, q)) for q in [10, 50, 90]]))
if len(refs) < 2:
    refs = [int(gc.min()), int(gc.max())]
size_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#888',
           markeredgecolor='white', markersize=np.sqrt(g * DOT_SIZE_SCALE),
           linewidth=0, label=f'{g}') for g in refs
]
ax.legend(handles=size_handles, title='Gene count', bbox_to_anchor=(1.02, 0.5),
          loc='center left', frameon=False, fontsize=8, title_fontsize=9)
ax.add_artist(leg1)

fig.subplots_adjust(right=RIGHT_PLOT_FRAC)
plt.savefig(FIG_DIR / 'Fig4c_GO_BP_bubble.pdf', bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved: {FIG_DIR / "Fig4c_GO_BP_bubble.pdf"}')